<a href="https://colab.research.google.com/github/giannayan/02_conditionals/blob/master/Titanic_Extra_Credit_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Titanic Survival Prediction using Logistic Regression

import math
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler



# load data
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
data = pd.read_csv(url)

# keep useful columns
data = data[['Pclass','Sex','Age','Fare','Survived']]


# make gender 0 (male) or 1 (female)
data['Sex'] = data['Sex'].map({'male':0,'female':1})

# drop missing values
data = data.dropna()

scaler = StandardScaler()
data[['Pclass','Sex','Age','Fare']] = scaler.fit_transform(
    data[['Pclass','Sex','Age','Fare']]
)

# insert bias
data.insert(0,"x0",1)

print("Dataset preview:\n")
print(data.head())

# Convert dataframe to list format
observations = data.values.tolist()

# split dataset into training and testing data
train, test = train_test_split(observations, test_size=0.3, random_state=0)

print("\nTraining examples:", len(train))
print("Test examples:", len(test))

# logistic regression
def sigmoid(n):
    return 1/(1+math.exp(-n))


def fit_model(observations, n_iterations=5000, step_size=0.00005):

    # initialize parameters
    theta = [0] * (len(observations[0]) - 1)

    for t in range(n_iterations):

        gradient_theta = [0] * len(theta)

        # compute gradient
        for obs in observations:

            x = obs[:-1]
            y = obs[-1]

            dot_prod = 0
            for i in range(len(theta)):
                dot_prod += theta[i] * x[i]

            for k in range(len(theta)):
                gradient_theta[k] += x[k] * (y - sigmoid(dot_prod))

        # update parameters
        for i in range(len(theta)):
            theta[i] += step_size * gradient_theta[i]

    return theta


# train model
theta = fit_model(train)

print("\nLearned weights:")
print(theta)


# test model
correct = 0
predicted_probs = []
actual_labels = []

for obs in test:

    x = obs[:-1]
    y = obs[-1]

    dot_prod = 0
    for i in range(len(theta)):
        dot_prod += theta[i] * x[i]

    pred_prob = sigmoid(dot_prod)

    if pred_prob >= 0.5:
        pred = 1
    else:
        pred = 0

    predicted_probs.append(pred_prob)
    actual_labels.append(y)

    if pred == y:
        correct += 1

accuracy = correct / len(test)

print("\nModel Accuracy:", accuracy)


# sample predictions
print("\nSample Predictions:\n")

for i in range(10):

    obs = test[i]

    x = obs[:-1]
    y = obs[-1]

    dot_prod = 0
    for j in range(len(theta)):
        dot_prod += theta[j] * x[j]

    prob = sigmoid(dot_prod)

    print("Actual:", y, " Predicted survival probability:", round(prob,3))


# feature weights
feature_names = ["bias","Passenger Class","Sex","Age","Fare"]

print("\nFeature Weights Interpretation:\n")

for i in range(len(theta)):
    print(feature_names[i], "weight =", round(theta[i],4))

print("""
Positive weight -> increases survival probability
Negative weight -> decreases survival probability
""")


# user input simulation
print("\nLet's predict survival probability for a passenger! 🚢🧊")

pclass = float(input("Passenger Class (1 = First, 2 = Second, 3 = Third): "))
sex = input("Sex (male/female): ").strip().lower()
age = float(input("Age: "))
fare = float(input("Ticket Fare: "))

# convert sex
sex_value = 0 if sex == "male" else 1

# scale the features

user_df = pd.DataFrame(
    [[pclass, sex_value, age, fare]],
    columns=['Pclass','Sex','Age','Fare']
)

scaled = scaler.transform(user_df)


pclass_s, sex_s, age_s, fare_s = scaled[0]

# build feature vector
x = [1, pclass_s, sex_s, age_s, fare_s]

dot_prod = 0
for i in range(len(theta)):
    dot_prod += theta[i] * x[i]

prob = sigmoid(dot_prod)

print("\nPredicted survival probability:", round(prob,3))

if prob >= 0.5:
    print("Prediction: Survived 🤠")
else:
    print("Prediction: Did NOT survive 💀")

Dataset preview:

   x0    Pclass       Sex       Age      Fare  Survived
0   1  0.911232 -0.759051 -0.530377 -0.518978         0
1   1 -1.476364  1.317434  0.571831  0.691897         1
2   1  0.911232  1.317434 -0.254825 -0.506214         1
3   1 -1.476364  1.317434  0.365167  0.348049         1
4   1  0.911232 -0.759051  0.365167 -0.503850         0

Training examples: 499
Test examples: 215

Learned weights:
[-0.5382184734746295, -1.021381745896793, 1.1823189761186235, -0.5015632835106633, 0.12115535611702338]

Model Accuracy: 0.8325581395348837

Sample Predictions:

Actual: 0.0  Predicted survival probability: 0.525
Actual: 0.0  Predicted survival probability: 0.86
Actual: 1.0  Predicted survival probability: 0.792
Actual: 0.0  Predicted survival probability: 0.195
Actual: 1.0  Predicted survival probability: 0.547
Actual: 0.0  Predicted survival probability: 0.069
Actual: 0.0  Predicted survival probability: 0.229
Actual: 0.0  Predicted survival probability: 0.253
Actual: 0.0  Pre